# Les modules et les imports en Python

Certaines fonctionnalités de Python ne sont pas chargées par défaut — il faut les **importer** sous forme de modules.

On distingue trois sources de modules :

1. **La bibliothèque standard** (*standard library*) — intégrée à Python, toujours disponible ([liste complète](https://docs.python.org/3/py-modindex.html)). Exemples : `math`, `os`, `json`, `datetime`, `pathlib`…
2. **Les packages tiers** — installés depuis [PyPI](https://pypi.org/) via `pip` ou `uv`. Exemples : `pandas`, `matplotlib`, `scikit-learn`…
3. **Vos propres modules** — des fichiers `.py` que vous créez dans votre projet.

## Module, package, librairie — quelle différence ?

| Terme | Définition | Exemple |
|-------|-----------|---------|
| **Module** | Un seul fichier `.py` contenant du code (fonctions, classes, variables) | `math`, `json`, un fichier `utils.py` que vous créez |
| **Package** | Un dossier contenant plusieurs modules, avec un fichier `__init__.py` | `matplotlib` (contient les sous-modules `pyplot`, `colors`, etc.) |
| **Librairie** (*library*) | Terme générique pour un package (ou ensemble de packages) distribué et installable | `pandas`, `scikit-learn`, `requests` |

> En pratique, on utilise souvent "librairie" et "package" de manière interchangeable — ce n'est pas grave.

---

## Les différentes formes d'import

### 1. Import simple

On importe le module entier. On accède à son contenu avec la notation `module.objet` :

In [ ]:
import math

math.cos(12)

### 2. Importer un objet spécifique d'un module

On importe directement une fonction (ou classe, variable) — plus besoin du préfixe :

In [ ]:
from math import cos

cos(12)

On peut importer plusieurs objets d'un coup :

In [ ]:
from math import cos, sin, pi

print(cos(pi))  # -1.0
print(sin(pi))  # ~0.0 (pas exactement 0 à cause de la précision flottante)

### 3. Importer avec un alias

On donne un nom plus court au module avec `as`. C'est la forme la plus courante en data science :

In [ ]:
import math as m

m.cos(12)

On peut aussi aliaser un objet spécifique (plus rare) :

In [ ]:
from math import degrees as deg

deg(3.14159)  # convertit des radians en degrés

### 4. Importer un sous-module

Certaines librairies sont organisées en sous-modules. L'exemple classique : `matplotlib.pyplot`.

In [ ]:
# La forme standard — avec un alias
import matplotlib.pyplot as plt

plt.plot([1, 2, 4]);

In [ ]:
# Forme alternative (équivalente)
from matplotlib import pyplot as plt

plt.plot([1, 2, 4]);

---

## Les alias conventionnels de la data

En data science, certains alias sont des **conventions universelles**. Tout le monde les utilise, tout le code en ligne les utilise, les LLM les génèrent automatiquement. À connaître par cœur :

In [ ]:
import numpy as np               # calcul numérique (tableaux, algèbre linéaire)
import pandas as pd              # manipulation de données tabulaires
import matplotlib.pyplot as plt  # graphiques
import seaborn as sns            # graphiques statistiques (basé sur matplotlib)
import plotly.express as px      # graphiques interactifs
import statsmodels.api as sm     # modèles statistiques

> **Ne pas utiliser d'autres alias** : écrire `import pandas as pds` ou `import numpy as n` est techniquement valide, mais ça rend le code confus pour tout le monde. Respectez les conventions.

---

## L'anti-pattern : `from module import *`

Il existe une forme d'import qui importe **tout** le contenu d'un module d'un coup :

In [ ]:
# ⚠️ NE FAITES PAS ÇA — c'est un exemple de ce qu'il ne faut PAS faire
# from math import *

# Pourquoi c'est un problème ?
# 1. On ne sait plus d'où viennent les fonctions
# 2. Ça peut écraser silencieusement des noms existants
# 3. Impossible de savoir ce qui a été importé sans lire le code source du module

# Exemple du problème :
# from math import *       # importe "log" (logarithme)
# from logging import *    # écrase "log" avec la fonction de logging !
# log(10)                  # → erreur incompréhensible

---

## Le `__init__.py` et ce qui se passe quand on importe un package

Quand vous écrivez `import pandas as pd`, Python exécute le fichier `__init__.py` qui se trouve à la racine du package `pandas`. C'est ce fichier qui décide **ce qui est directement accessible** depuis `pd.` :

```python
import pandas as pd

pd.DataFrame(...)     # ✅ disponible — importé dans le __init__.py de pandas
pd.read_csv(...)      # ✅ disponible
pd.io.formats.style   # un sous-module interne — pas directement exposé
```

C'est pour cela qu'on peut écrire `pd.DataFrame` directement, sans avoir besoin de `from pandas.core.frame import DataFrame`.

Les auteurs de la librairie font un travail de **design d'API** pour exposer les objets les plus utiles au premier niveau, et cacher les détails d'implémentation dans les sous-modules.

In [ ]:
import pandas as pd

# pd.DataFrame est directement accessible grâce au __init__.py
df = pd.DataFrame({"nom": ["Alice", "Bob"], "age": [30, 25]})
df

---

## Imports relatifs vs absolus

Quand vous verrez du code généré par un LLM ou dans un projet open source, vous rencontrerez deux styles d'import :

```python
# Import absolu — le chemin complet depuis la racine du projet/package
from mon_projet.utils.nettoyage import nettoyer_texte

# Import relatif — le chemin relatif au fichier courant (le . signifie "ce dossier")
from .utils.nettoyage import nettoyer_texte
from ..autre_module import ma_fonction  # .. = dossier parent
```

Les imports relatifs (avec des `.`) ne fonctionnent **que dans un package** (un dossier avec un `__init__.py`). Dans un notebook ou un script isolé, on utilise toujours des imports absolus.

> **En pratique pour vous** : utilisez des imports absolus. Si vous voyez des imports relatifs dans du code, c'est que le code fait partie d'un package structuré.

---

## Récapitulatif

| Forme | Syntaxe | Quand l'utiliser |
|-------|---------|-----------------|
| Import simple | `import math` | Quand on utilise plusieurs fonctions du module |
| Import spécifique | `from math import cos` | Quand on n'a besoin que d'un ou deux objets |
| Import avec alias | `import pandas as pd` | La norme en data science — alias conventionnels |
| Import de sous-module | `import matplotlib.pyplot as plt` | Pour les librairies organisées en sous-modules |
| Import étoile | `from math import *` | **Jamais** (sauf dans un shell interactif pour tester vite) |

**Bonnes pratiques** :
- Les imports sont toujours **en haut du fichier/notebook**
- Un import par ligne (ou un `from X import a, b, c` si c'est le même module)
- Utilisez les alias conventionnels (`pd`, `np`, `plt`, `sns`…)
- Évitez `import *`

---

**Pour aller plus loin** : la logique de ce qui est exposé ou non dans l'API d'une librairie est un vrai sujet de design. Pour ceux que ça intéresse, l'article sur le design de l'API de scikit-learn est une lecture fascinante : [API design for machine learning software (Buitinck et al., 2013)](https://arxiv.org/abs/1309.0238).